# 因子表 Fold 切分

对 `factortb.parquet` 按时间顺序生成多折训练/测试集切分，输出 CSV 到 `output/{date}/` 目录。

## Step 0 — 参数配置

### 该 Step 涉及参数

| 参数 | 来源 | 用途 |
|------|------|------|
| `PARQUET_PATH` | 数据路径 | 因子表 parquet 文件位置 |
| `LOOKBACK_YEARS` | 用户设定 | 训练集回看年数 |
| `INITIAL_TRAIN_END` | 用户设定 | 第一折训练集结束日期 |
| `GAP_DAYS` | 用户设定 | 训练集与测试集间隔交易日数 |
| `FOLD_STEP_MONTHS` | 用户设定 | 每折推进月数 |
| `LABEL_COL` | 数据列名 | 预测目标列 |
| `OUTPUT_ROOT` | 用户设定 | CSV 输出根目录 |

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# ===== 参数区 =====
PARQUET_PATH = Path("D:/quant/ML_teamwork/data/factortb.parquet")
LOOKBACK_YEARS = 4
INITIAL_TRAIN_END = "2020-01-01"
GAP_DAYS = 10
FOLD_STEP_MONTHS = 6
LABEL_COL = "FWD_RET_10D"
OUTPUT_ROOT = Path("D:/quant/ML_teamwork/00_fold切分/01_train&test/output")

print("参数配置完成")
print(f"  数据: {PARQUET_PATH}")
print(f"  回看年数: {LOOKBACK_YEARS}    初始训练截止: {INITIAL_TRAIN_END}")
print(f"  间隔交易日: {GAP_DAYS}    推进月数: {FOLD_STEP_MONTHS}")
print(f"  标签列: {LABEL_COL}")


参数配置完成
  数据: D:\quant\ML_teamwork\data\factortb.parquet
  回看年数: 4    初始训练截止: 2020-01-01
  间隔交易日: 10    推进月数: 6
  标签列: FWD_RET_10D


## Step 1 — 函数定义

### 函数清单

| 函数 | 用途 |
|------|------|
| `load_factor_table` | 加载 parquet，排序，重置索引 |
| `identify_columns` | 识别 ALPHA_* 特征列和 FWD_RET_* 标签列 |
| `_snap_left` | 日期向左对齐到最近交易日 |
| `_snap_right` | 日期向右对齐到最近交易日 |
| `_shift_trade_days` | 交易日偏移 |
| `generate_folds` | 生成 fold 定义列表 |
| `split_fold_df` | 按 fold 定义切分 train/test DataFrame |

In [3]:
def load_factor_table(parquet_path: Path) -> pd.DataFrame:
    """加载因子表 parquet, 按 trade_date + ts_code 排序。

    输入:
        parquet_path: parquet 文件路径 (Path)

    输出:
        pd.DataFrame: 排序后的因子表, trade_date 已归一化, index 已重置
    """
    df = pd.read_parquet(parquet_path)
    df["trade_date"] = pd.to_datetime(df["trade_date"]).dt.normalize()
    df = df.sort_values(["trade_date", "ts_code"], kind="mergesort").reset_index(drop=True)
    return df


def identify_columns(df: pd.DataFrame) -> Tuple[List[str], List[str]]:
    """识别特征列 (ALPHA_*) 和标签列 (FWD_RET_*), 并校验特征列不含未来数据。

    输入:
        df: 因子表 DataFrame

    输出:
        (feature_cols, label_cols): 特征列名列表, 标签列名列表
    """
    feature_cols = [c for c in df.columns if c.startswith("ALPHA_")]
    label_cols   = [c for c in df.columns if c.startswith("FWD_RET_")]
    assert all(not c.startswith("FWD_RET_") for c in feature_cols), "特征列含未来收益字段!"
    return feature_cols, label_cols


def _snap_left(trade_dates: pd.DatetimeIndex, ts: pd.Timestamp) -> Optional[pd.Timestamp]:
    """日期向左对齐——找到 <= ts 的最近一个交易日。

    输入:
        trade_dates: 交易日序列 (pd.DatetimeIndex)
        ts: 待对齐的时间戳

    输出:
        最近的交易日 (pd.Timestamp), 若无则 None
    """
    idx = int(trade_dates.searchsorted(ts, side="right")) - 1
    return None if idx < 0 else trade_dates[idx]


def _snap_right(trade_dates: pd.DatetimeIndex, ts: pd.Timestamp) -> Optional[pd.Timestamp]:
    """日期向右对齐——找到 >= ts 的最近一个交易日。

    输入:
        trade_dates: 交易日序列 (pd.DatetimeIndex)
        ts: 待对齐的时间戳

    输出:
        最近的交易日 (pd.Timestamp), 若无则 None
    """
    idx = int(trade_dates.searchsorted(ts, side="left"))
    return None if idx >= len(trade_dates) else trade_dates[idx]


def _shift_trade_days(trade_dates: pd.DatetimeIndex, anchor: pd.Timestamp, n_shift: int) -> Optional[pd.Timestamp]:
    """从 anchor 偏移 n_shift 个交易日。

    输入:
        trade_dates: 交易日序列 (pd.DatetimeIndex)
        anchor: 锚点时间戳
        n_shift: 偏移交易日数 (正=向后, 负=向前)

    输出:
        偏移后的交易日 (pd.Timestamp), 越界则 None
    """
    idx = int(trade_dates.searchsorted(anchor, side="left"))
    target = idx + n_shift
    return None if (target < 0 or target >= len(trade_dates)) else trade_dates[target]


def generate_folds(
    trade_dates: np.ndarray,
    lookback_years: int,
    initial_train_end: str,
    gap_days: int,
    step_months: int,
) -> List[Dict]:
    """按步进月份生成多折训练/测试日期范围 (walk-forward 切分)。

    输入:
        trade_dates: 所有交易日 (np.ndarray)
        lookback_years: 训练集回看年数
        initial_train_end: 第一折训练集结束日期 (str, "YYYY-MM-DD")
        gap_days: 训练/测试间隔交易日数 (防信息泄漏)
        step_months: 每折推进月数

    输出:
        folds: list[dict], 每折含 fold/train_start/train_end/test_start/test_end
    """
    td_index = pd.DatetimeIndex(pd.to_datetime(trade_dates)).sort_values()
    last_td, first_td = td_index[-1], td_index[0]
    initial_end_ts = pd.Timestamp(initial_train_end)

    folds: List[Dict] = []
    k = 0
    while True:
        train_end_candidate = initial_end_ts + pd.DateOffset(months=k * step_months)
        test_end_candidate  = initial_end_ts + pd.DateOffset(months=(k + 1) * step_months)

        train_end_ts = _snap_left(td_index, train_end_candidate)
        test_end_ts  = _snap_left(td_index, test_end_candidate)
        if train_end_ts is None or test_end_ts is None:
            break
        if test_end_candidate > last_td:
            break

        train_start_candidate = train_end_ts - pd.DateOffset(years=lookback_years)
        if train_start_candidate < first_td:
            k += 1
            continue
        train_start_ts = _snap_right(td_index, train_start_candidate)
        if train_start_ts is None or train_start_ts > train_end_ts:
            k += 1
            continue

        test_start_ts = _shift_trade_days(td_index, train_end_ts, gap_days + 1)
        if test_start_ts is None or test_start_ts > test_end_ts:
            break

        folds.append({
            "fold":        k + 1,
            "train_start": train_start_ts.date(),
            "train_end":   train_end_ts.date(),
            "test_start":  test_start_ts.date(),
            "test_end":    test_end_ts.date(),
        })
        k += 1
    return folds


def split_fold_df(df: pd.DataFrame, fold: Dict, label_col: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """按 fold 定义切分训练/测试集, 去除标签缺失行。

    输入:
        df: 全量因子表 (pd.DataFrame)
        fold: fold 定义 dict, 含 train_start/train_end/test_start/test_end
        label_col: 标签列名, 用于 dropna

    输出:
        (train_df, test_df): 训练集和测试集 (均已按 trade_date + ts_code 排序)
    """
    tr_s = pd.Timestamp(fold["train_start"]); tr_e = pd.Timestamp(fold["train_end"])
    te_s = pd.Timestamp(fold["test_start"]);  te_e = pd.Timestamp(fold["test_end"])

    train_df = df[(df["trade_date"] >= tr_s) & (df["trade_date"] <= tr_e)].copy()
    test_df  = df[(df["trade_date"] >= te_s) & (df["trade_date"] <= te_e)].copy()
    train_df = (train_df.dropna(subset=[label_col])
                .sort_values(["trade_date", "ts_code"]).reset_index(drop=True))
    test_df  = (test_df.dropna(subset=[label_col])
                .sort_values(["trade_date", "ts_code"]).reset_index(drop=True))
    return train_df, test_df


print("[OK] Step 1 函数定义完成")

[OK] Step 1 函数定义完成


## Step 2 — 加载数据

### 该 Step 涉及参数

| 参数 | 来源 | 用途 |
|------|------|------|
| `PARQUET_PATH` | Step 0 | 数据文件路径 |
| `LABEL_COL` | Step 0 | 标签列名 |

In [4]:
df = load_factor_table(PARQUET_PATH)
_dmin = df["trade_date"].min().date()
_dmax = df["trade_date"].max().date()
print(f"全表 shape: {df.shape}    日期: {_dmin} ~ {_dmax}")

feature_cols, label_cols = identify_columns(df)
assert LABEL_COL in label_cols, f"LABEL_COL '{LABEL_COL}' 不在数据列中"
print(f"特征 ALPHA_*: {len(feature_cols)} 列    标签 FWD_RET_*: {len(label_cols)} 列 -> {label_cols}")

trade_dates = np.sort(df["trade_date"].unique())
print(f"交易日数: {len(trade_dates)}")

_label = LABEL_COL
print(f"标签 '{_label}' 缺失率: {df[_label].isna().mean():.4%}")


全表 shape: (883816, 233)    日期: 2014-01-02 ~ 2026-05-22
特征 ALPHA_*: 230 列    标签 FWD_RET_*: 1 列 -> ['FWD_RET_10D']
交易日数: 3007
标签 'FWD_RET_10D' 缺失率: 0.3474%


## Step 3 — 生成 Fold 计划

### 该 Step 涉及参数

| 参数 | 来源 | 用途 |
|------|------|------|
| `LOOKBACK_YEARS` | Step 0 | 训练集回看年数 |
| `INITIAL_TRAIN_END` | Step 0 | 第一折训练集结束日期 |
| `GAP_DAYS` | Step 0 | 训练/测试间隔交易日 |
| `FOLD_STEP_MONTHS` | Step 0 | 每折推进月数 |

In [5]:
folds = generate_folds(
    trade_dates=trade_dates,
    lookback_years=LOOKBACK_YEARS,
    initial_train_end=INITIAL_TRAIN_END,
    gap_days=GAP_DAYS,
    step_months=FOLD_STEP_MONTHS,
)
fold_ids = [f["fold"] for f in folds]
print(f"\n生成 fold 数: {len(folds)}    fold_ids={fold_ids}")
_hdr = f"{'fold':>4} | {'train_start':>10} ~ {'train_end':>10} | {'test_start':>10} ~ {'test_end':>10}"
print(_hdr)
print("-" * 70)
for f in folds:
    _fid = f["fold"]
    _trs = str(f["train_start"])
    _tre = str(f["train_end"])
    _tes = str(f["test_start"])
    _tee = str(f["test_end"])
    print(f"{_fid:>4} | {_trs:>10} ~ {_tre:>10} | {_tes:>10} ~ {_tee:>10}")



生成 fold 数: 12    fold_ids=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
fold | train_start ~  train_end | test_start ~   test_end
----------------------------------------------------------------------
   1 | 2015-12-31 ~ 2019-12-31 | 2020-01-16 ~ 2020-07-01
   2 | 2016-07-01 ~ 2020-07-01 | 2020-07-16 ~ 2020-12-31
   3 | 2017-01-03 ~ 2020-12-31 | 2021-01-18 ~ 2021-07-01
   4 | 2017-07-03 ~ 2021-07-01 | 2021-07-16 ~ 2021-12-31
   5 | 2018-01-02 ~ 2021-12-31 | 2022-01-18 ~ 2022-07-01
   6 | 2018-07-02 ~ 2022-07-01 | 2022-07-18 ~ 2022-12-30
   7 | 2019-01-02 ~ 2022-12-30 | 2023-01-17 ~ 2023-06-30
   8 | 2019-07-01 ~ 2023-06-30 | 2023-07-17 ~ 2023-12-29
   9 | 2019-12-30 ~ 2023-12-29 | 2024-01-16 ~ 2024-07-01
  10 | 2020-07-01 ~ 2024-07-01 | 2024-07-16 ~ 2024-12-31
  11 | 2020-12-31 ~ 2024-12-31 | 2025-01-16 ~ 2025-07-01
  12 | 2021-07-01 ~ 2025-07-01 | 2025-07-16 ~ 2025-12-31


## Step 4 — 切分并输出 CSV

### 该 Step 涉及参数

| 参数 | 来源 | 用途 |
|------|------|------|
| `OUTPUT_ROOT` | Step 0 | CSV 输出根目录 |
| `LABEL_COL` | Step 0 | 用于 dropna 的标签列 |

### Step 输出

- `output/{date}/fold_{k}_train.csv` — 第 k 折训练集
- `output/{date}/fold_{k}_test.csv` — 第 k 折测试集
- `output/{date}/fold_plan.csv` — fold 计划总表

In [ ]:
import datetime as _dt
_run_date = _dt.date.today().strftime("%Y%m%d")
_out_dir = OUTPUT_ROOT / _run_date
_out_dir.mkdir(parents=True, exist_ok=True)
print(f"输出目录: {_out_dir}")

# 保存 fold 计划
plan_df = pd.DataFrame(folds)
plan_path = _out_dir / "fold_plan.csv"
plan_df.to_csv(plan_path, index=False)
print(f"[SAVED] {plan_path}")

_label = LABEL_COL
for f in folds:
    _fid = f["fold"]
    train_df, test_df = split_fold_df(df, f, _label)
    
    tr_path = _out_dir / f"fold_{_fid}_train.csv"
    te_path = _out_dir / f"fold_{_fid}_test.csv"
    train_df.to_csv(tr_path, index=False)
    test_df.to_csv(te_path, index=False)
    
    _trs = str(f["train_start"])
    _tre = str(f["train_end"])
    _tes = str(f["test_start"])
    _tee = str(f["test_end"])
    print(f"fold {_fid:>2} | train: {train_df.shape[0]:>7} rows  {_trs}~{_tre} | test: {test_df.shape[0]:>7} rows  {_tes}~{_tee}")

print()
print(f"[DONE] 全部 {len(folds)} 折输出完成 -> {_out_dir}")


输出目录: D:\quant\ML_teamwork\00_fold切分\01_train&test\output\20260526
[SAVED] D:\quant\ML_teamwork\00_fold切分\01_train&test\output\20260526\fold_plan.csv
fold  1 | train:  282719 rows  2015-12-31~2019-12-31 | test:   32331 rows  2020-01-16~2020-07-01
fold  2 | train:  283629 rows  2016-07-01~2020-07-01 | test:   34446 rows  2020-07-16~2020-12-31
fold  3 | train:  285212 rows  2017-01-03~2020-12-31 | test:   32646 rows  2021-01-18~2021-07-01
fold  4 | train:  287213 rows  2017-07-03~2021-07-01 | test:   34167 rows  2021-07-16~2021-12-31
fold  5 | train:  288890 rows  2018-01-02~2021-12-31 | test:   32379 rows  2022-01-18~2022-07-01
fold  6 | train:  290172 rows  2018-07-02~2022-07-01 | test:   34171 rows  2022-07-18~2022-12-30
fold  7 | train:  291142 rows  2019-01-02~2022-12-30 | test:   32396 rows  2023-01-17~2023-06-30
fold  8 | train:  291252 rows  2019-07-01~2023-06-30 | test:   34200 rows  2023-07-17~2023-12-29
fold  9 | train:  291329 rows  2019-12-30~2023-12-29 | test:   32395 rows 